# Broker Auth: Fyers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivikasavnish/algo-trading-notebooks/blob/main/notebooks/05_broker_auth_fyers.ipynb)

Authenticate and place a demo order via the FYERS API over your static IP.

Part 05 of 35 in the [ServLoci algo/options trading notebook series](https://comm.servloci.in/docs) — full index in `notebooks/README.md`.

## Setup

In [ ]:
# Get your dedicated static IPv6 + SOCKS5 credentials free:
#   https://comm.servloci.in/register        (or /auth/google?free=1 for an instant trial)
# Your api_key / api_secret pair shows up in the portal after signup:
#   https://comm.servloci.in/user
!pip install -q "requests[socks]"
!curl -sL https://comm.servloci.in/sdk/servloci.py -o servloci.py

import os
from servloci import ServLoci

SERVLOCI_API_KEY = os.environ.get("SERVLOCI_API_KEY", "dhan:1000000001")   # broker:client_id
SERVLOCI_API_SECRET = os.environ.get("SERVLOCI_API_SECRET", "")            # from the portal — leave blank to run this notebook in demo mode

sl = None
if SERVLOCI_API_SECRET:
    sl = ServLoci(api_key=SERVLOCI_API_KEY, api_secret=SERVLOCI_API_SECRET)
    print("ServLoci configured:", sl.host, sl.port)
else:
    print("SERVLOCI_API_SECRET not set — running in demo mode (no live proxy calls).")

**Support level:** Confirmation flow for individual and family accounts — FYERS support verifies the requesting IP before enabling API trading.

Indian broker APIs authenticate trading sessions with an OAuth-style
two-step handshake, not a single long-lived key. You redirect the user (in
practice, yourself) to a broker login page, the broker redirects back with a
short-lived `request_token`, and you exchange that token — together with your
app's `api_secret` — for an `access_token` that's valid for actual order
calls. The `api_key`/`api_secret` pair identifies *your app*; the
`access_token` identifies *an authenticated session* and, at every broker
covered here, expires at end of trading day regardless of activity. That
forced daily re-login is a deliberate session-security control, common across
SEBI-regulated broker APIs, so a stolen access token has a shelf life measured
in hours, not indefinitely.

FYERS sits between Kite's instant self-service and Dhan's ticket queue: API
access is enabled per app, but support verifies the requesting IP before
switching it on, including for family/sub-accounts trading under one
umbrella. FYERS also publishes a WebSocket feed alongside the REST API, which
matters once you move from polling (notebook 11) to a lower-latency data
path.

In [ ]:
if sl:
    sl.attach()  # BEFORE importing fyers-apiv3 — it creates a requests.Session at import time
    from fyers_apiv3 import fyersModel

    fyers = fyersModel.FyersModel(client_id=os.environ.get("FYERS_CLIENT_ID", ""),
                                   token=os.environ.get("FYERS_ACCESS_TOKEN", ""), is_async=False)
    print("Fyers client ready")
    # order = fyers.place_order(data={"symbol": "NSE:INFY-EQ", "qty": 1, "type": 2,
    #                                  "side": 1, "productType": "INTRADAY"})
else:
    print("Demo mode — set SERVLOCI_API_SECRET above to run this against your Fyers account.")

Install the broker SDK separately: `!pip install -q fyers-apiv3`. Full whitelist steps: [https://comm.servloci.in/docs/brokers](https://comm.servloci.in/docs).

---

« Previous: [Broker Auth: Groww](04_broker_auth_groww.ipynb)  
Next: [Black-Scholes Pricing & Greeks](06_black_scholes_pricing_greeks.ipynb) »

Try the concepts above interactively: [Options Strategy Builder](https://comm.servloci.in/tools/strategy-builder) · [Docs](https://comm.servloci.in/docs) · [Get your static IP](https://comm.servloci.in/register)